In [1]:
import sys
!{sys.executable} -m pip install -i https://mirror-pypi.runflare.com/simple scikit-learn imbalanced-learn

Looking in indexes: https://mirror-pypi.runflare.com/simple


In [2]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipe    # instead of sklearn.pipeline.Pipeline
                                                    # to support SMOTE process

from sklearn.preprocessing import StandardScaler, FunctionTransformer

import joblib

In [3]:
BASE_DIR = Path.cwd().parents[0]    # 'Trained Model' folder
ROOT = BASE_DIR.parent    # 'Diabetes Impact Simulator' folder

sys.path.append(str(ROOT/'diabetes_simulator'))
# Python can only import modules from locations listed in sys.path

from diabetes_predictor.preprocessing import handle_zeros

df = pd.read_csv(BASE_DIR/'diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


# Pipeline steps

In [4]:
# Wrap the function so sklearn can use it
cleaner = FunctionTransformer(handle_zeros)

pipeline = ImbPipe([
    ('cleaner', cleaner),
    ('smote', SMOTE(random_state=7)),    
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=500,
        penalty='l2',
        C=0.1,
        random_state=7
    ))
])

# Splitting

In [5]:
df = df.sample(frac=1, random_state=7)
X = df.drop(columns=['Outcome'])
y = df['Outcome']

feature_names = X.columns.tolist()   # we need this in 'extract_params.ipynb' to coordinate coeffs with features

x_tr, x_ts, y_tr, y_ts = train_test_split(X, y, test_size=0.2, stratify=y, random_state=7)

# Training the Pipeline

In [6]:
pipeline.fit(x_tr, y_tr)

# fit() makes all steps in pipeline to be applied to the training data only (to learn and transform it)

preds = pipeline.predict(x_ts)

# by predict(), pipeline applies all the preprocessing steps (that have been fitted on the training data) to the test data, EXCEPT 'SMOTE'

print("Default threshold (0.5):")
print(f"Accuracy:  {accuracy_score(y_ts, preds):.4f}")
print(f"Recall:    {recall_score(y_ts, preds):.4f}")
print(f"Precision: {precision_score(y_ts, preds):.4f}")
print(f"F1:        {f1_score(y_ts, preds):.4f}")

probs = pipeline.predict_proba(x_ts)[:, 1]
threshold = 0.45
tuned_preds = (probs >= threshold).astype(int)
print(f"\nTuned threshold ({threshold}):")
print(f"Accuracy:  {accuracy_score(y_ts, tuned_preds):.4f}")
print(f"Recall:    {recall_score(y_ts, tuned_preds):.4f}")
print(f"Precision: {precision_score(y_ts, tuned_preds):.4f}")
print(f"F1:        {f1_score(y_ts, tuned_preds):.4f}")

Default threshold (0.5):
Accuracy:  0.7403
Recall:    0.6852
Precision: 0.6167
F1:        0.6491

Tuned threshold (0.45):
Accuracy:  0.7273
Recall:    0.7593
Precision: 0.5857
F1:        0.6613


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
F:\Network Science\Project\Diabetes Impact Simulator\diabetes_simulator\diabetes_predictor\preprocessing.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '115.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[zero_mask, col] = col_med
F:\Network Science\Project\Diabetes Impact Simulator\diabetes_simulator\diabetes_predictor\preprocessing.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. V

# Exporting

In [7]:
joblib.dump(pipeline, 'diabetes_pipeline.pkl')
joblib.dump(threshold, 'logreg_threshold.pkl')

print("Files 'diabetes_pipeline.pkl' & 'logreg_threshold.pkl' exported.")

Files 'diabetes_pipeline.pkl' & 'logreg_threshold.pkl' exported.


In [6]:
joblib.dump(feature_names, 'feature_names.pkl')

print("File 'feature_names.pkl' exported.")

File 'feature_names.pkl' exported.
